# Geometry-only clustering review

This notebook does not select a winner automatically. A clustering result is first screened for minimum cluster size and cross-example support; supported candidates are then compared through several metrics, stability, and physical interpretation.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'RAPID').exists():
    REPO_ROOT = REPO_ROOT.parents[1]
OUTPUT_ROOT = Path('/Volumes/PhD/river_hierarchy/network_variants/output')
EXAMPLE_IDS = ['02', '03', '05', '11', '13', '15', '21', '56']
RUN_DIR = REPO_ROOT / 'RAPID' / 'classification' / 'results' / f"clustering-review-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"


In [ ]:
command = [sys.executable, str(REPO_ROOT / 'RAPID/classification/train_rapid_geometry_clustering.py'),
           '--output-root', str(OUTPUT_ROOT), '--results-dir', str(RUN_DIR),
           '--example-ids', *EXAMPLE_IDS]
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
summary = pd.read_csv(RUN_DIR / 'clustering_quality_summary.csv')
regular = summary.loc[summary.valid_exploratory_solution].copy()
outliers = summary.loc[summary.outlier_candidate].copy()
unsupported = summary.loc[~summary.valid_exploratory_solution & ~summary.outlier_candidate].copy()
print('Regular-structure candidates:')
display(regular)
print('Outlier candidates — inspect separately; they are not automatically rejected or treated as regular classes:')
display(outliers)
print('Unsupported partitions:')
display(unsupported)


In [ ]:
ALGORITHM_TO_INSPECT = 'kmeans_k2'  # select a candidate from the tables above
assignments = pd.read_csv(RUN_DIR / 'cluster_assignments.csv')
plot_df = assignments.loc[assignments.algorithm.eq(ALGORITHM_TO_INSPECT)].copy()
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(plot_df.len_fraction_width_weighted_log_lambda, plot_df.lag_to_inflow_peak_pct_shift_vs_base, c=plot_df.cluster, cmap='tab10', alpha=.8)
ax.axhline(0, color='0.4', linewidth=1, linestyle='--')
ax.set(xlabel='Selected-domain length fraction × −log10(Lambda_K) × width fraction', ylabel='Lag-to-inflow-peak shift vs base [%]', title=f'{ALGORITHM_TO_INSPECT}: RAPID response shown after geometry-only clustering')
fig.colorbar(scatter, ax=ax, label='Geometry-only cluster')
plt.show()
display(plot_df.groupby('cluster').agg(n=('state_id','size'), examples=('example_id','nunique'), response_mean=('lag_to_inflow_peak_pct_shift_vs_base','mean'), response_median=('lag_to_inflow_peak_pct_shift_vs_base','median')))
